In [1]:
import nltk
import numpy as np
import re
import shutil
import tensorflow as tf
import os
import unicodedata
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [12]:

tf.random.set_seed(123)
np.random.seed(123)

NameError: name 'tf' is not defined

# Load Data

In [16]:
import re
import unicodedata

def preprocess_sentence(sent):
    # Normalize the input sentence to decompose accented characters into their base form
    sent = "".join([c for c in unicodedata.normalize("NFD", sent) if unicodedata.category(c) != "Mn"])
    
    # Add space before punctuation marks like '.', '!', or '?' to tokenize them separately
    sent = re.sub(r"([!.?])", r" \1", sent)
    
    # Replace any sequence of characters that are not letters or punctuation with a single space
    sent = re.sub(r"[^a-zA-Z!.?]+", r" ", sent)
    
    # Replace multiple consecutive spaces with a single space
    sent = re.sub(r"\s+", " ", sent)
    
    # Convert the sentence to lowercase to ensure consistency in case
    sent = sent.lower()
    return sent

In [ ]:
def read_data(num_sent_pairs =20000):
    en_sents, fr_sents_in, fr_sents_out = [], [], []
    local_file = os.path.join("datasets", "fra.txt")
    with open(local_file, "r") as fin:
        for i, line in enumerate(fin):
            en_sent, fr_sent, _ = line.strip().split('\t')
            en_sent = [w for w in preprocess_sentence(en_sent).split()]
            fr_sent = preprocess_sentence(fr_sent)
            fr_sent_in = [w for w in ("BOS " + fr_sent).split()]
            fr_sent_out = [w for w in (fr_sent + " EOS").split()]
            en_sents.append(en_sent)
            fr_sents_in.append(fr_sent_in)
            fr_sents_out.append(fr_sent_out)
            if i >= num_sent_pairs - 1:
                break
    return en_sents, fr_sents_in, fr_sents_out

In [14]:
import pandas as pd

def read_data(file_path, num_sent_pairs=20000):
    en_sents, id_sents_in, id_sents_out = [], [], []
    
    df = pd.read_csv(file_path)
    df = df[["translate_en", "text_processed"]].dropna().head(num_sent_pairs)

    for _, row in df.iterrows():
        en = preprocess_sentence(row["translate_en"])  # Target = English
        idn = preprocess_sentence(row["text_processed"])  # Source = Indonesia

        en_sent = [w for w in en.split()]
        id_sent = [w for w in idn.split()]

        # Untuk bahasa target (English), tambahkan BOS dan EOS
        en_in = ["BOS"] + en_sent
        en_out = en_sent + ["EOS"]

        en_sents.append(id_sent)     # input/source (Indo)
        id_sents_in.append(en_in)    # decoder input (BOS + Eng)
        id_sents_out.append(en_out)  # decoder target (Eng + EOS)

    return en_sents, id_sents_in, id_sents_out


In [21]:
NUM_SENT_PAIRS = 1000
id_sents, en_sents_in, en_sents_out = read_data("translated_google.csv",NUM_SENT_PAIRS)

In [22]:
print(id_sents[0:5])

[['amit', 'amit', '.', '.', '.', 'kejadian', 'kayak', 'begini', 'bikin', 'dia', 'besar', 'kepala', 'memalukan'], ['presiden', 'prabowo', 'bakal', 'pidato', 'poltik', 'di', 'may', 'day', 'monas', 'naik', 'maung', 'dari', 'istana', 'prabowo', 'hariburuh', 'mayday'], ['lepaskan', 'hidup', 'dari', 'dendam'], ['sayang', 'nya', 'pak', 'presiden', 'subianto', 'tidak', 'merasa', 'terhina', 'ketika', 'preman', 'menghina', 'purnawirawan', 'bau', 'tanah', 'sekelas', 'pak', 'sutyoso', 'pendidikan', 'militer', 'dan', 'pencapaian', 'nya', 'luar', 'biasa', 'seenak', 'nya', 'di', 'hina', 'oleh', 'preman', 'di', 'biarkan', 'oleh', 'presiden', 'seakan', 'situasi', 'ini', 'sengaja', 'diciptakan', '?'], ['heran', '.', '.kok', 'bisa', 'balas', 'dendam', 'ya', '.', '.pdhl', 'dia', 'udah', 'pensiun', 'berarti', 'masih', 'tetap', 'pegang', 'kendali', 'pemerintahan', 'yang', '.dipimin', 'prabowo']]


In [23]:
print(en_sents_in[0:5])

[['BOS', 'amit', 'amit', '.', '.', '.', 'events', 'like', 'this', 'make', 'him', 'big', 'shameful', 'head'], ['BOS', 'president', 'prabowo', 'will', 'be', 'a', 'poltic', 'speech', 'at', 'may', 'day', 'monas', 'on', 'maung', 'from', 'prabowo', 'hariburuh', 'mayday', 'palace'], ['BOS', 'let', 'go', 'of', 'life', 'from', 'revenge'], ['BOS', 'unfortunately', 'mr', '.', 'president', 'subianto', 'did', 'not', 'feel', 'insulted', 'when', 'thugs', 'insulted', 'retirement', 'of', 'the', 'smell', 'of', 'land', 'of', 'the', 'same', 'class', 'as', 'the', 'military', 'education', 'and', 'his', 'achievements', 'were', 'extraordinary', 'as', 'insulted', 'by', 'thugs', 'were', 'allowed', 'by', 'the', 'president', 'as', 'if', 'this', 'situation', 'was', 'deliberately', 'created', '?'], ['BOS', 'surprised', '.', '.', '.', 'you', 'can', 'get', 'revenge', 'huh', '.', '.', '.', 'but', 'he', 'has', 'retired', 'means', 'still', 'still', 'holding', 'the', 'control', 'of', 'the', 'government', '.', 'dipimin', 

In [24]:
print(en_sents_out[0:5])

[['amit', 'amit', '.', '.', '.', 'events', 'like', 'this', 'make', 'him', 'big', 'shameful', 'head', 'EOS'], ['president', 'prabowo', 'will', 'be', 'a', 'poltic', 'speech', 'at', 'may', 'day', 'monas', 'on', 'maung', 'from', 'prabowo', 'hariburuh', 'mayday', 'palace', 'EOS'], ['let', 'go', 'of', 'life', 'from', 'revenge', 'EOS'], ['unfortunately', 'mr', '.', 'president', 'subianto', 'did', 'not', 'feel', 'insulted', 'when', 'thugs', 'insulted', 'retirement', 'of', 'the', 'smell', 'of', 'land', 'of', 'the', 'same', 'class', 'as', 'the', 'military', 'education', 'and', 'his', 'achievements', 'were', 'extraordinary', 'as', 'insulted', 'by', 'thugs', 'were', 'allowed', 'by', 'the', 'president', 'as', 'if', 'this', 'situation', 'was', 'deliberately', 'created', '?', 'EOS'], ['surprised', '.', '.', '.', 'you', 'can', 'get', 'revenge', 'huh', '.', '.', '.', 'but', 'he', 'has', 'retired', 'means', 'still', 'still', 'holding', 'the', 'control', 'of', 'the', 'government', '.', 'dipimin', 'prabow

In [29]:
import tensorflow as tf

# We declare the tokenizers and use them to transform texts to sequences of indices

tokenizer_id = tf.keras.preprocessing.text.Tokenizer(filters="", lower=False)
print(f'tokenizer_id: {tokenizer_id}')
tokenizer_id.fit_on_texts(id_sents)
data_id = tokenizer_id.texts_to_sequences(id_sents)
print(f'data en: {data_id[0:5]}')
data_id = tf.keras.preprocessing.sequence.pad_sequences(data_id, padding="post")
print(f'data en: \n{data_id[0:5]}')

tokenizer_en = tf.keras.preprocessing.text.Tokenizer(filters="", lower=False)
tokenizer_en.fit_on_texts(en_sents_in)
tokenizer_en.fit_on_texts(en_sents_out)
data_en_in = tokenizer_en.texts_to_sequences(en_sents_in)
data_en_in = tf.keras.preprocessing.sequence.pad_sequences(data_en_in, padding="post")
data_en_out = tokenizer_en.texts_to_sequences(en_sents_out)
data_en_out = tf.keras.preprocessing.sequence.pad_sequences(data_en_out, padding="post")

tokenizer_id: <keras.preprocessing.text.Tokenizer object at 0x0000021C0A981790>
data en: [[588, 588, 1, 1, 1, 589, 184, 226, 120, 28, 185, 161, 1004], [12, 5, 121, 590, 1005, 7, 81, 82, 404, 405, 591, 19, 227, 5, 1006, 291], [1007, 186, 19, 592], [593, 42, 8, 12, 139, 14, 140, 1008, 406, 11, 141, 98, 292, 228, 407, 8, 1009, 594, 408, 3, 1010, 42, 187, 229, 1011, 42, 7, 1012, 83, 11, 7, 1013, 83, 12, 595, 409, 6, 410, 1014, 4], [1015, 1, 1016, 16, 596, 592, 25, 1, 1017, 28, 111, 597, 142, 74, 188, 598, 293, 189, 2, 1018, 5]]
data en: 
[[ 588  588    1    1    1  589  184  226  120   28  185  161 1004    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0]
 [  12    5  121  590 1005    7   81   82  404  405  591   19  227    5
  1006  291    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0 

In [30]:
# We then build up dictionaries and vocabularies

vocab_size_id = len(tokenizer_id.word_index)
vocab_size_en = len(tokenizer_en.word_index)

word2idx_id = tokenizer_id.word_index
idx2word_id = {v:k for k, v in word2idx_id.items()}
word2idx_en = tokenizer_en.word_index
idx2word_en = {v:k for k, v in word2idx_en.items()}

print("vocab size (en): {:d}, vocab size (fr): {:d}".format(vocab_size_id, vocab_size_en))
maxlen_id = data_id.shape[1]
maxlen_en = data_en_out.shape[1]
print("seqlen (en): {:d}, (fr): {:d}".format(maxlen_id, maxlen_en))

print(f'\nword2idx_id: \n\n{word2idx_id}')

vocab size (en): 2922, vocab size (fr): 2535
seqlen (en): 54, (fr): 68

word2idx_id: 

{'.': 1, 'yang': 2, 'dan': 3, '?': 4, 'prabowo': 5, 'ini': 6, 'di': 7, 'pak': 8, '!': 9, 'itu': 10, 'preman': 11, 'presiden': 12, 'ada': 13, 'tidak': 14, 'indonesia': 15, 'bisa': 16, 'ke': 17, 'orang': 18, 'dari': 19, 'jokowi': 20, 'negara': 21, 'jadi': 22, 'dengan': 23, 'apa': 24, 'ya': 25, 'sudah': 26, 'karena': 27, 'dia': 28, 'ormas': 29, 'ga': 30, 'gak': 31, 'untuk': 32, 'aja': 33, 'rakyat': 34, 'tppbersamamenteridesa': 35, 'bukan': 36, 'tapi': 37, 'akan': 38, 'hercules': 39, 'kalo': 40, 'mau': 41, 'nya': 42, 'saja': 43, 'anda': 44, 'lebih': 45, 'lagi': 46, 'ijazah': 47, 'tni': 48, 'para': 49, 'mereka': 50, 'kalau': 51, 'jangan': 52, 'sama': 53, 'juga': 54, 'punya': 55, 'merah': 56, 'putih': 57, 'semua': 58, 'harus': 59, 'gibran': 60, 'dalam': 61, 'musdesus': 62, 'pembentukan': 63, 'koperasi': 64, 'palsu': 65, 'grib': 66, 'atau': 67, 'buat': 68, 'dulu': 69, 'kita': 70, 'pernah': 71, 'hanya': 72, 

In [31]:
# Convert to dataset format

batch_size = 64
dataset = tf.data.Dataset.from_tensor_slices((data_id, data_en_in, data_en_out))
dataset = dataset.shuffle(10000)
test_size = NUM_SENT_PAIRS // 4
test_dataset = dataset.take(test_size).batch(batch_size, drop_remainder=True)
train_dataset = dataset.skip(test_size).batch(batch_size, drop_remainder=True)

print(train_dataset)

<BatchDataset element_spec=(TensorSpec(shape=(64, 54), dtype=tf.int32, name=None), TensorSpec(shape=(64, 68), dtype=tf.int32, name=None), TensorSpec(shape=(64, 68), dtype=tf.int32, name=None))>


In [32]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, num_timesteps, encoder_dim, **kwargs):
        super(Encoder, self).__init__(**kwargs)
        self.encoder_dim = encoder_dim
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=num_timesteps)
        self.rnn = tf.keras.layers.GRU(encoder_dim, return_sequences=False, return_state=True)
    
    def call(self, x, state):
        x = self.embedding(x)
        x, state = self.rnn(x, initial_state=state) # x is output, and state is hidden state
        return x, state
    
    def init_state(self, batch_size):
        return tf.zeros((batch_size, self.encoder_dim))

In [33]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, num_timesteps, decoder_dim, **kwargs):
        super(Decoder, self).__init__(**kwargs)
        self.decoder_dim = decoder_dim
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length= num_timesteps)
        self.rnn = tf.keras.layers.GRU(decoder_dim, return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, x, state):
        x = self.embedding(x)
        x, state = self.rnn(x, state)
        x = self.dense(x) # only return logits
        return x, state

In [34]:
embedding_dim = 256
encoder_dim, decoder_dim = 1024, 1024

# vocab size + 1 for word that is not in the library usually called as out-of-vocabulary (OOV)
encoder = Encoder(vocab_size_id+1, embedding_dim, maxlen_id, encoder_dim)
decoder = Decoder(vocab_size_en+1, embedding_dim, maxlen_en, decoder_dim)

In [35]:
encoder_in, decoder_in, decoder_out = next(iter(train_dataset))

encoder_state = encoder.init_state(batch_size)
encoder_out, encoder_state = encoder(encoder_in, encoder_state)
decoder_state = encoder_state
decoder_pred, decoder_state = decoder(decoder_in, decoder_state)

print("encoder input :", encoder_in.shape)
print("encoder output :", encoder_out.shape, "state:", encoder_state.shape)
print("decoder output (logits):", decoder_pred.shape, "state:", decoder_state.shape)
print("decoder output (labels):", decoder_out.shape)

encoder input : (64, 54)
encoder output : (64, 1024) state: (64, 1024)
decoder output (logits): (64, 68, 2536) state: (64, 1024)
decoder output (labels): (64, 68)


In [36]:
def loss_fn(ytrue, ypred):
    scce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    mask = tf.math.logical_not(tf.math.equal(ytrue, 0))
    mask = tf.cast(mask, dtype=tf.int64)
    loss = scce(ytrue, ypred, sample_weight=mask)
    return loss

In [37]:
@tf.function
def train_step(encoder_in, decoder_in, decoder_out, encoder_state):
    with tf.GradientTape() as tape:
        encoder_out, encoder_state = encoder(encoder_in, encoder_state)
        decoder_state = encoder_state
        decoder_pred, decoder_state = decoder(decoder_in, decoder_state)
        loss = loss_fn(decoder_out, decoder_pred)
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return loss

In [38]:
def predict(encoder, decoder, batch_size, sents_id, data_id, sents_en_out, word2idx_en, idx2word_en):
    random_id = np.random.choice(len(sents_id))
    print("input : ", " ".join(sents_id[random_id]))
    print("label : ", " ".join(sents_en_out[random_id]))
    encoder_in = tf.expand_dims(data_id[random_id], axis=0)
    decoder_out = tf.expand_dims(sents_en_out[random_id], axis=0)
    
    encoder_state = encoder.init_state(1)
    encoder_out, encoder_state = encoder(encoder_in, encoder_state)
    
    decoder_state = encoder_state
    decoder_in = tf.expand_dims(tf.constant([word2idx_en["BOS"]]), axis=0)
    
    pred_sent_fr = []
    decoding_step = 0
    while decoding_step < maxlen_en:
        decoder_pred, decoder_state = decoder(decoder_in, decoder_state)
        decoder_pred = tf.argmax(decoder_pred, axis=-1)
        pred_word = idx2word_en[decoder_pred.numpy()[0][0]]
        pred_sent_fr.append(pred_word)
        if pred_word == "EOS":
            break
        decoder_in = decoder_pred
        decoding_step += 1
    print("predicted: ", " ".join(pred_sent_fr))

In [42]:
import os
import numpy as np

checkpoint_dir = "./checkpoints"
optimizer = tf.keras.optimizers.Adam()
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(optimizer=optimizer, encoder=encoder, decoder=decoder)
num_epochs = 100
eval_scores = []
for e in range(num_epochs):
    encoder_state = encoder.init_state(batch_size)
    for batch, data in enumerate(train_dataset):
        encoder_in, decoder_in, decoder_out = data
        # print(encoder_in.shape, decoder_in.shape, decoder_out.shape)
        loss = train_step(encoder_in, decoder_in, decoder_out, encoder_state)
        # print("Batch {}: loss = {}".format(batch, loss))
    print("Epoch: {}, Loss: {:.4f}".format(e + 1, loss.numpy()))
    if e % 10 == 0:
        checkpoint.save(file_prefix=checkpoint_prefix)
    predict(encoder, decoder, batch_size, id_sents, data_id, en_sents_out, word2idx_en, idx2word_en)

checkpoint.save(file_prefix=checkpoint_prefix)

Epoch: 1, Loss: 1.6605
input :  ini karena pemerintah memberi ruang dan lemah dalam menindak premanisme atau mungkin memang d jadikan alat kekuasaan
label :  this is because the government gives space and is weak in cracking down on thuggery or maybe it is a means of power EOS
predicted:  the culprit of the red and white cooperative EOS
Epoch: 2, Loss: 1.4110
input :  lu tau arti buzzer gasi ? dimana letak kebuzzeran gua ?
label :  do you know the meaning of a buzzer ?where is my kebuzzeran ? EOS
predicted:  the original of the red and white cooperative tppbersamamenteridesa EOS
Epoch: 3, Loss: 1.4483
input :  gimana hercules masih disayang mimin ormas menjanjikan ya anggota kalian preman emang ga malu kalian dikadalinpreman ganti baju aj preman jd polisi polisi jd anak buah preman gimana
label :  how do hercules still loved by mimin ormas promising yes your members thugs are not ashamed you guys are channeled to change clothes so the police are the police EOS
predicted:  the president

UnknownError: {{function_node __wrapped__SaveV2_dtypes_12_device_/job:localhost/replica:0/task:0/device:CPU:0}} Failed to WriteFile: ./checkpoints\ckpt-6_temp/part-00000-of-00001.data-00000-of-00001.tempstate9886409439813425643 : There is not enough space on the disk.
; operation in progress [Op:SaveV2]